# Decision-threshold optimization

Show explicitly that model fitting and maintenance decisions are different optimization problems.

In [ ]:
from pathlib import Path

from scania_aps.data import TEST_FILENAME, TRAIN_FILENAME, read_raw_csv

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"
train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)
print(train.X.shape, test.X.shape, train.y.mean(), test.y.mean())

In [ ]:
import numpy as np

from scania_aps.costs import bayes_threshold, maintenance_cost, optimize_threshold
from scania_aps.models.logistic import LogisticConfig, build_logistic_pipeline
from scania_aps.split import research_split

split = research_split(train.X, train.y)
model = build_logistic_pipeline(LogisticConfig(penalty="l2", C=0.1)).fit(split.X_fit, split.y_fit)
probs = model.predict_proba(split.X_threshold)[:, 1]
opt = optimize_threshold(split.y_threshold.to_numpy(), probs)
for threshold in [0.5, bayes_threshold(), opt.threshold]:
    cost = maintenance_cost(split.y_threshold.to_numpy(), (probs >= threshold).astype(np.int8))
    print(threshold, cost)